# Peta Program — dari data sensor sampai keputusan

**Riset Deteksi Fault 4 Sensor Kelembaban**

Notebook ini menjawab satu pertanyaan: *program ini sebenarnya melakukan apa,
urutannya bagaimana, dan angkanya diambil dari mana?*

Cara membaca: cukup jalankan semua sel dari atas ke bawah. Tidak ada pelatihan
model di sini, jadi selesai dalam hitungan detik.

Tiga notebook sidang:

| Notebook | Isi | Waktu jalan |
|---|---|---|
| **00 Peta Program** (ini) | alur, glosarium, peta angka | < 5 detik |
| **01 Alur Sistem** | contoh data nyata di setiap tahap | < 2 menit |
| **02 Hasil** | tabel dan grafik hasil akhir | < 30 detik |

## 1 — Alur program dalam satu gambar

Delapan kotak di bawah adalah urutan kerja program. Kotak 1–4 menyiapkan data,
kotak 5 mengubah sinyal menjadi angka, kotak 6–7 menebak, kotak 8 menguji.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

stages = [
    ("1. Data mentah", "4 sensor, tiap 30 detik"),
    ("2. Broker", "4 kanal disatukan, tidak dirata-rata"),
    ("3. Sinkron waktu", "satu grid tiap 5 menit"),
    ("4. Injeksi fault", "drift, spike, bias, macet"),
    ("5. Window + entropi", "potong 16,7 jam, ukur kerumitan"),
    ("6. ANN", "60 angka -> kelas fault"),
    ("7. Rantai sensor", "fault -> sensor mana"),
    ("8. Uji", "5 skenario x 3 ukuran window"),
]

fig, ax = plt.subplots(figsize=(13, 3.2))
ax.set_xlim(0, 8); ax.set_ylim(0, 1); ax.axis("off")
for i, (title, sub) in enumerate(stages):
    box = FancyBboxPatch((i + 0.06, 0.30), 0.88, 0.42,
                         boxstyle="round,pad=0.02,rounding_size=0.05",
                         facecolor="#e8f1fb", edgecolor="#2c6ea8", lw=1.4)
    ax.add_patch(box)
    ax.text(i + 0.5, 0.58, title, ha="center", va="center", fontsize=8.5, weight="bold")
    ax.text(i + 0.5, 0.42, sub, ha="center", va="center", fontsize=7)
    if i < len(stages) - 1:
        ax.annotate("", xy=(i + 1.01, 0.51), xytext=(i + 0.94, 0.51),
                    arrowprops=dict(arrowstyle="->", color="#2c6ea8", lw=1.2))
ax.set_title("Alur kerja program (kiri ke kanan)", fontsize=11, weight="bold")
plt.tight_layout(); plt.show()

## 2 — Tiap tahap dalam bahasa sehari-hari

| Tahap | Yang dilakukan | Kenapa perlu |
|---|---|---|
| **1. Data mentah** | Baca `data_sensor.csv` (281.721 baris, 14 Sep–21 Des 2025) | bahan dasar |
| **2. Broker** | Empat kolom sensor disatukan, **identitas tiap sensor tetap** | supaya bisa tahu sensor mana yang rusak |
| **3. Sinkron waktu** | Rata-rata 5 menit, buang duplikat, tambal slot bolong | semua sensor bicara di jam yang sama |
| **4. Injeksi fault** | Tambahkan gangguan buatan sesuai skenario | memberi label "rusak" yang pasti |
| **5. Window + entropi** | Potong jadi jendela 200 sampel, ukur kerumitan pola | model butuh angka, bukan grafik mentah |
| **6. ANN** | 60 angka masuk, keluar kelas fault | menebak jenis kondisi |
| **7. Rantai sensor** | Dari tebakan fault, lacak sensor mana | tindakan perbaikan jadi terarah |
| **8. Uji** | 5 skenario × 3 ukuran window, CV berkelompok | angka tidak boleh menipu diri sendiri |

## 3 — Enam angka yang paling sering ditanya

| Pertanyaan | Jawaban | Sumber |
|---|---|---|
| Metode mana lebih tinggi? | JSD-Fuzzy menang **11 dari 15 sel**, rata-rata **+0,028** | `11` §10 |
| Sensor mana yang dikenali? | F1 **0,675 vs 0,634**, menang 9/15 | `11` §12 |
| Jenis fault tersulit? | **bias** (0,448), lalu spike (0,601) | `05` |
| Kalau sensor diidentifikasi dari hasil klasifikasi? | F1 **0,902 vs 0,846** | `04` |
| Pelatih ANN terbaik? | **BR 0,429 ≈ LM 0,433** (p = 0,81) | `09` |
| ANN vs XGBoost? | Fitur JSD+Time rata-rata **0,750**; XGBoost juga kuat | `06` |

**Satu kalimat statistik.** Selisih +0,028 konsisten, tetapi uji gabungan belum
kuat (pooled p = 0,076); rincian ada di lampiran privat.

> Yang tidak diklaim: "JSD terbukti signifikan lebih baik". Yang diklaim:
> "JSD konsisten lebih tinggi pada 11 dari 15 sel".

## 4 — Peta notebook: sidang ↔ penelitian

Tabel ini supaya tidak bingung kalau penguji minta "notebook aslinya".

| Bagian sidang | Notebook penelitian | Kernel |
|---|---|---|
| Alur & diagram | `07` §0–§9 | `jsd-fuzzy-cv-5skenario` v23 |
| Angka utama | `11` §10–§13 | `jsd-edm-ann-lm-lengkap-q3` v3 |
| Broker 4 kanal | `02` | `jsd-fuzzy-broker-architecture-v2` v3 |
| Data mentah | `Ilustrasi` | `ilustrasi-data-akuisisi-dan-fault` v3 |
| Jenis fault | `05` | `jsd-fuzzy-faulttype-per-sensor` v2 |
| Two-stage | `04` | `jsd-fuzzy-twostage-sensorid` v6 |
| Pelatih ANN | `09` | `ann-lm-vs-bayesian-vs-ga-paper-pmdc` v3 |
| ANN vs XGBoost | `06` | `jsd-xgb-perscenario` v4 |

Tabel lengkap (nomor sel + nama file): `sidang/MAPPING.md`.

**Peringatan penting.** Notebook `01/04/05/07/08` memakai `MLPClassifier`
sklearn, **bukan** Levenberg–Marquardt dari paper. Yang benar-benar LM adalah
`09`, `10`, `11`. Jangan menyebut "LM" untuk notebook lain.

## 5 — Glosarium 10 istilah

| Istilah | Arti sederhana |
|---|---|
| **Fault** | kerusakan sensor: drift, spike, bias, atau macet |
| **Drift** | nilai bergeser pelan ke satu arah |
| **Spike** | loncatan sesaat lalu kembali |
| **Bias** | nilai bergeser tetap, tidak kembali |
| **Hardware malfunction** | sensor macet atau kehilangan data |
| **Window** | potongan sinyal sepanjang 200 sampel (±16,7 jam) |
| **Entropi** | ukuran "seberapa tidak beraturan" pola sinyal |
| **Multiskala** | entropi dihitung pada 15 tingkat kekasaran |
| **EDM-Fuzzy** | kemiripan pola memakai jarak garis lurus |
| **JSD-Fuzzy** | kemiripan pola memakai bentuk sebaran nilai |

## 6 — Kalau penguji bertanya, buka ini

| Pertanyaan | Buka |
|---|---|
| "Perlihatkan datanya" | `01 Alur Sistem` bagian 1 |
| "Kenapa tidak dirata-rata?" | `01` bagian 2, lalu `02` |
| "Contoh sinyal rusak?" | `01` bagian 3 |
| "Kenapa 60 fitur?" | `01` bagian 5 |
| "Hasilnya mana?" | `02 Hasil` bagian 1 |
| "Sensor mana yang rusak?" | `02 Hasil` bagian 2 |
| "Signifikan?" | `02 Hasil` bagian 7 |
| "Batasan?" | `02 Hasil` bagian 8 |

---
**Status:** privat · commit `27737ee` · 2026-10-26
**Rujukan lengkap:** `sidang/MAPPING.md` · **Bahan tanya-jawab:** `Bahan_Sidang.md`